# QLoRA для ассистента ректора (Qwen2.5-14B)

**Что делает ноутбук.** Дообучает Qwen2.5-14B-Instruct на парах
«вопрос → ответ по фрагменту» из базы знаний мониторинга и выгружает
GGUF q4_k_m для Ollama.

**Чего он не делает.** Он не «вшивает» цифры мониторинга в модель.
Цифры остаются в RAG: они меняются каждый год, и проверяемость важнее
памяти. LoRA учит формат ответа, ссылку на источник и отказ, когда
данных нет.

**Перед запуском.** В Kaggle добавьте датасет `train.jsonl`
(собирается `python docs/kaggle/dataset_build.py`) и включите GPU T4.


## 1. Окружение

In [ ]:
# Unsloth ставится отдельно; версии фиксируем, чтобы ноутбук
# не сломался от обновления зависимостей.
!pip install -q "unsloth[cu121-torch240] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
import torch
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM, ГБ:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


## 2. Модель и токенизатор

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-14B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)


## 3. LoRA-адаптер

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


## 4. Данные

Формат — готовые `messages` (system / user / assistant). Важно, что в
каждом примере есть фрагмент: модель учится опираться на него, а не на
свою память.


In [ ]:
import json, pathlib
from datasets import Dataset

path = pathlib.Path("/kaggle/input/monitoring-qa/train.jsonl")
if not path.exists():
    # путь может отличаться: найдите файл и поправьте
    matches = list(pathlib.Path("/kaggle/input").rglob("train.jsonl"))
    assert matches, "train.jsonl не найден в /kaggle/input"
    path = matches[0]

rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
dataset = Dataset.from_list(rows)
print("примеров:", len(dataset))
print(dataset[0]["messages"][1]["content"][:300])


def to_text(row):
    return {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(to_text, remove_columns=dataset.column_names)


## 5. Обучение

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="/kaggle/working/out",
        report_to="none",
    ),
)
stats = trainer.train()
print(stats)


## 6. Выгрузка в GGUF

Unsloth сам собирает GGUF — отдельная сборка llama.cpp не нужна.
`q4_k_m` — компромисс: ~9 ГБ и почти без потери качества.


In [ ]:
model.save_pretrained_gguf(
    "/kaggle/working/gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
!ls -lh /kaggle/working/gguf


## 7. Проверка перед скачиванием

Сначала смотрим, что модель отвечает по фрагменту, и только потом
скачиваем 9 ГБ.


In [ ]:
FastLanguageModel.for_inference(model)

question = "Какое место занимает ИУБиП по объёму НИОКР среди вузов области в 2025?"
context = "5. ИУБиП — 419,85 ← ИУБиП"

messages = [
    {"role": "system", "content": "Ты — ассистент ректора. Отвечай по фрагменту и указывай источник."},
    {"role": "user", "content": f"Вопрос: {question}\n\nФрагмент:\n{context}"},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=160, temperature=0.1)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))


## 8. Что делать с файлом

1. Скачать `gguf_gguf/qwen2.5-14b-instruct.Q4_K_M.gguf` из `/kaggle/working`.
2. На сервере положить рядом `Modelfile`:

```
FROM ./qwen2.5-14b-instruct.Q4_K_M.gguf
PARAMETER temperature 0.2
PARAMETER num_ctx 4096
SYSTEM Ты — ассистент ректора Южного университета (ИУБиП). Отвечай по-русски, кратко, со ссылкой на документ. Если данных нет — скажи об этом.
```

3. `ollama create rector-qwen -f Modelfile`
4. `python -m backend.models small rector-qwen`
5. Сравнить с базовой моделью: `python -m backend.smoke --ask "..."`.

**Скорость не изменится**: это та же 14B, а наша ВМ считает без AVX2.
